# Test ELT Process

## Environment Setup & Mock Data Generation

In [2]:
# Safe magic usage so this cell can run as a script or in a notebook  # noqa: EXE002
import sqlite3
import sys
from pathlib import Path

import pandas as pd
from openai import OpenAI
from prefect.blocks.system import Secret

repo_root = Path.cwd().parent.resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))
from src.configs import root_path
from src.elt.ingestion import execute_ingestion
from src.elt.scanner import scan_sources
from src.elt.strategy import generate_strategy
from src.state.schema import get_unified_staging_view, init_schema

try:
    ip = get_ipython()
except NameError:
    ip = None

if ip is not None:
    ip.run_line_magic('load_ext', 'autoreload')
    ip.run_line_magic('autoreload', '2')

db_path = repo_root / "data" / "ledger.db"
okf_path = repo_root / "knowledge" / ".okf"

In [5]:
# Create Mock Directories for Testing
mock_gdrive = root_path("notebooks/mock_gdrive")
mock_onedrive = root_path("notebooks/mock_onedrive")
mock_nextcloud = root_path("notebooks/mock_nextcloud")

for d in [mock_gdrive, mock_onedrive, mock_nextcloud]:
    d.mkdir(parents=True, exist_ok=True)

# Generate Mock Files (Notice 'tax_2026.pdf' is exactly the same in both drives!)
(mock_gdrive / "tax_2026.pdf").write_text("DUMMY PDF CONTENT: TAX 2026")
(mock_gdrive / "aws_invoice_july.txt").write_text("AWS BILL: $150")

(mock_onedrive / "tax_2026.pdf").write_text("DUMMY PDF CONTENT: TAX 2026") # EXACT DUPLICATE
(mock_onedrive / "family_photo.jpg").write_text("DUMMY JPEG CONTENT: BEACH")

print("✅ Mock Cloud Drives and Files Generated.")

✅ Mock Cloud Drives and Files Generated.


## Phase 0 - Sync DB with Physical State

In [4]:
import sqlite3
from pathlib import Path

import pandas as pd

from src.configs import load_settings, root_path
from src.state.schema import sync_production_inventory_from_disk

settings = load_settings()
test_db = root_path("data/test_ledger.db")
nextcloud_mount = Path("/nextcloud_data") # Or root_path("notebooks/mock_nextcloud") for mock testing

# Run the Sync
sync_production_inventory_from_disk(test_db, nextcloud_mount)

# Verify the populated table
conn = sqlite3.connect(test_db)
df = pd.read_sql_query("SELECT id, blake3_hash, nextcloud_path, file_size, ingested_at FROM production_inventory", conn)
conn.close()

print(f"\n📊 Total active records in production_inventory: {len(df)}")
display(df)

🔍 Syncing production_inventory from physical disk: /nextcloud_data/admin/files
   [Adopted] /Media/Photos/Family/family_photo.jpg
   [Adopted] /Media/Photos/Family/photo.jpg
   [Adopted] /Templates/Ecosystem.odp
   [Adopted] /Templates/Report.odt
   [Adopted] /Templates/Meeting agenda.whiteboard
   [Adopted] /Templates/Menu.odt
   [Adopted] /Templates/Product plan.md
   [Adopted] /Templates/Invoice.ods
   [Adopted] /Templates/Modern company.odp
   [Adopted] /Templates/Business model canvas.odg
   [Adopted] /Templates/Simple.odp
   [Adopted] /Templates/Invoice.odt
   [Adopted] /Templates/Mindmap.odg
   [Adopted] /Templates/Business model canvas.ods
   [Adopted] /Templates/Mother's day.odt
   [Adopted] /Templates/Business model canvas.whiteboard
   [Adopted] /Templates/Party invitation.odt
   [Adopted] /Templates/Meeting notes.md
   [Adopted] /Templates/Impact effort.whiteboard
   [Adopted] /Templates/Brainstorming.whiteboard
   [Adopted] /Templates/Kanban board.whiteboard
   [Adopted] /

    id                                        blake3_hash  \
0   12  42bda40860bf4172587d265575376d6bdd656e02fcefba...   
1   13  53f34198c1152cadab64c5745d9ecff100a9f1cf0aa961...   
2   14  11d4c1d28d03f2e57c14aa767ee293dee25cc128f4ac28...   
3   15  88f600727045ceaaee21c922e1d8534d239c15b62e15c2...   
4   16  5d7c69cee22ab6301759f2dd4a1795b7996e6bf270a54a...   
5   17  40687a89bc1a57ed112df27f76973532029452de055959...   
6   18  3dd552359a7c4f7c60347e31fa884b9c84bc7bb9c90a5d...   
7   19  638471da0cab8f3fada4ab324f6831c94b6c220afea6fa...   
8   20  8955a91169b3b7cdf697817a642026f58da46192452343...   
9   21  c6f307258a2d48feb78d6bfbdbf921c4e3ee613b9a009b...   
10  22  596d24758bdfe38235446a04873f149056ba7f95844dca...   
11  23  e28fa9d9a29a416f6de2b8ba45b74bffc2c42834128bbb...   
12  24  dc2e9802a990f9550dcd95d7a76657a38ed2915a9e7e6f...   
13  25  b2e999ecff560d7219159fdda307f665df906cbe30c0e8...   
14  26  609f2d94c31a7f004c5ef5ed11f3da04514950a322b441...   
15  27  33cc43f7ddf7adc6

## Phase 1 - Multi-Source Staging

In [6]:
# Define our sources
cloud_sources = {
    "Google Drive": str(mock_gdrive),
    "OneDrive": str(mock_onedrive)
}

# Use a test database so we don't mess up your real ledger
test_db = root_path("data/test_ledger.db")

print("--- PHASE 1: MULTI-SOURCE SCANNING ---")
init_schema(test_db, list(cloud_sources.keys()))
scan_sources(test_db, cloud_sources)

# Verify Staging Isolation
conn = sqlite3.connect(test_db)
print("\n📊 Staging Table: Google Drive")
display(pd.read_sql_query("SELECT original_path, sha256_hash, status FROM staging_google_drive", conn))

print("\n📊 Staging Table: OneDrive")
display(pd.read_sql_query("SELECT original_path, sha256_hash, status FROM staging_onedrive", conn))
conn.close()

--- PHASE 1: MULTI-SOURCE SCANNING ---
Schema initialized. Staging tables created for: ['Google Drive', 'OneDrive']
🔍 Scanning source: Google Drive at /home/coder/projects/agentic-nas-workflow/notebooks/mock_gdrive...


OperationalError: table staging_google_drive has no column named blake3_hash

## Phase 2 - Cross-Table Dedupe & Agentic Strategy

In [5]:
print("--- PHASE 2: CROSS-TABLE DEDUPLICATION & STRATEGY ---")

# Test SQL Deduplication
unified_view = get_unified_staging_view(test_db, list(cloud_sources.keys()))
print(f"\nUnique Files to Route: {len(unified_view)} (Expected 3, because the tax PDF is a duplicate!)")
display(pd.DataFrame(unified_view))

# Test Agentic Routing
print("\n🧠 Authenticating with Prefect Vault...")
llm_key = await Secret.load("gemini-api-key")
llm_client = OpenAI(api_key=llm_key.get(), base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

print("🤖 Asking Agent to generate taxonomy strategy...")
routings = generate_strategy(test_db, list(cloud_sources.keys()), llm_client, "gemini-2.5-pro")

print("\n🎯 Agentic Routing Strategy:")
for r in routings:
    print(f" - [Staging ID {r.staging_id}] -> {r.proposed_path}")
    print(f"   Reasoning: {r.reasoning}\n")

--- PHASE 2: CROSS-TABLE DEDUPLICATION & STRATEGY ---

Unique Files to Route: 3 (Expected 3, because the tax PDF is a duplicate!)


   staging_id          source_table  \
0           2      staging_onedrive   
1           1  staging_google_drive   
2           2  staging_google_drive   

                                       original_path              filename  \
0  /home/coder/projects/agentic-nas-workflow/note...      family_photo.jpg   
1  /home/coder/projects/agentic-nas-workflow/note...          tax_2026.pdf   
2  /home/coder/projects/agentic-nas-workflow/note...  aws_invoice_july.txt   

   file_size                                        sha256_hash  
0         25  180f3e2be00a7c4d3e12669804ad471d566e2b39801916...  
1         27  31b269b7404b2a6dd5ea19ed5f059f4047b85881595daf...  
2         14  84bb32d054844a0353a6f1b5ba4e2fa2f26c6d96797273...  


🧠 Authenticating with Prefect Vault...
🤖 Asking Agent to generate taxonomy strategy...
🧠 Asking Agent to route 3 unique files...

🎯 Agentic Routing Strategy:
 - [Staging ID 2] -> /Media/Photos/Family/family_photo.jpg
   Reasoning: The file is a family photograph, which fits directly into the existing /Media/Photos/Family taxonomy.

 - [Staging ID 1] -> /Documents/Financial/Taxes/tax_2026.pdf
   Reasoning: The file is a tax document from 2026, which belongs in the /Documents/Financial/Taxes directory.

 - [Staging ID 2] -> /Documents/Financial/Invoices/aws_invoice_july.txt
   Reasoning: The file is an invoice, which should be placed in the /Documents/Financial/Invoices directory.



## Phase 3 - Physical Ingestion & Production State

In [6]:
print("--- PHASE 3: PHYSICAL INGESTION ---")

# Execute the actual v5 ingestion engine!
# It natively uses shutil.copy2 to move files to our mock_nextcloud ZFS mount
execute_ingestion(
    db_path=test_db, 
    nextcloud_mount=mock_nextcloud, 
    routings=routings
)

# Verify the Production Inventory Source of Truth
conn = sqlite3.connect(test_db)
print("\n📊 Final Production Inventory:")
display(pd.read_sql_query("SELECT sha256_hash, nextcloud_path, file_size FROM production_inventory", conn))

# Verify Staging Tables were marked as 'ingested' or 'duplicate'
print("\n🔍 Final Google Drive Staging State:")
display(pd.read_sql_query("SELECT original_path, status FROM staging_google_drive", conn))

print("\n🔍 Final OneDrive Staging State:")
display(pd.read_sql_query("SELECT original_path, status FROM staging_onedrive", conn))

conn.close()

--- PHASE 3: PHYSICAL INGESTION ---
🚀 Ingesting: family_photo.jpg -> /Media/Photos/Family/family_photo.jpg
🚀 Ingesting: tax_2026.pdf -> /Documents/Financial/Taxes/tax_2026.pdf
🚀 Ingesting: aws_invoice_july.txt -> /Documents/Financial/Invoices/aws_invoice_july.txt

✅ Ingestion Complete | Success: 3 | Errors: 0

⚠️  ZFS Physical Moves Detected! ⚠️
To ensure Nextcloud sees these new files, you MUST run the OCC scanner on your TrueNAS host:
sudo docker exec -u www-data nextcloud php occ files:scan --all


📊 Final Production Inventory:


                                         sha256_hash  \
0  180f3e2be00a7c4d3e12669804ad471d566e2b39801916...   
1  31b269b7404b2a6dd5ea19ed5f059f4047b85881595daf...   
2  84bb32d054844a0353a6f1b5ba4e2fa2f26c6d96797273...   

                                      nextcloud_path  file_size  
0              /Media/Photos/Family/family_photo.jpg         25  
1            /Documents/Financial/Taxes/tax_2026.pdf         27  
2  /Documents/Financial/Invoices/aws_invoice_july...         14  


🔍 Final Google Drive Staging State:


                                       original_path    status
0  /home/coder/projects/agentic-nas-workflow/note...  ingested
1  /home/coder/projects/agentic-nas-workflow/note...  ingested


🔍 Final OneDrive Staging State:


                                       original_path    status
0  /home/coder/projects/agentic-nas-workflow/note...   pending
1  /home/coder/projects/agentic-nas-workflow/note...  ingested

In [3]:
from src.elt.ingestion import trigger_nextcloud_occ_scan

print("--- TESTING PREFECT/NOTEBOOK OCC SCAN TRIGGER ---")

# Trigger the scan directly from Python
success = trigger_nextcloud_occ_scan(container_name="ix-nextcloud-nextcloud-1")

if success:
    print("\n🎉 Refresh your Nextcloud Web UI (http://192.168.1.55:30027). All new files are now visible!")

--- TESTING PREFECT/NOTEBOOK OCC SCAN TRIGGER ---
✅ Nextcloud OCC Scan completed via Zero-Trust Proxy!

🎉 Refresh your Nextcloud Web UI (http://192.168.1.55:30027). All new files are now visible!
